# Multi-run Table 3: seed sweep with ± uncertainties

This runs the ActivitySCOPE pipeline from *More Complicated Workbook - including paper-specific items* `N_RUNS` times on **one copy of the orbit database loaded once**. Only the cross-validation fold assignment and the AutoGluon model seeds change between runs (run *k* uses seed `BASE_SEED + k`). The spread across runs is therefore the model's own run-to-run variability, not day-to-day database changes.

For every run it saves, as soon as the run finishes:
- `run_XXX_final.parquet`: the full `final` frame, with the ΔH sweep columns filled for the rows it was computed on (NaN elsewhere)
- `run_XXX_cometmerge.parquet`: the comet-file frame, with its own ΔH sweep columns
- `run_XXX_table_rows.csv`: the Table 3 values for this run (written last, so it marks the run as complete)
- `run_XXX_meta.json`: seed, timings, validation scores

With `RESUME = True`, re-running the notebook skips completed runs, so a crash overnight costs at most one run. The last section averages all completed runs into Table 3 with mean ± standard deviation for $P(N_{\rm opp}\ge4)$, $\Delta Q$, $S_{\rm EV}$ and $\Delta H$.

The pessimistic second pass (`prob_pessimistic`) is not included.

In [ ]:
import os
import gc
import re
import glob
import json
import time
import shutil
import datetime
import tempfile
import traceback
import importlib

import numpy as np
import pandas as pd
from scipy.stats import poisson
from autogluon.tabular import TabularPredictor
import torch
from IPython.display import display

import activityscope_utils as utils
import paper_table
importlib.reload(utils)
importlib.reload(paper_table)

<module 'paper_table' from '/Users/petervw/GitHub/ActivitySCOPE/paper_table.py'>

In [ ]:
# PARAMETERS

# Number of training runs and the base seed. Run k uses seed BASE_SEED + k for both the 8-fold
# assignment and the AutoGluon model seeds, so any single run can be reproduced on its own.
N_RUNS = 1
BASE_SEED = 1000

# Per-run outputs and the aggregated table go here. Change RUN_NAME to start a fresh sweep.
RUN_NAME = "table3_seed_sweep"
OUTPUT_DIR = os.path.join("multi_run_outputs", RUN_NAME)

# Skip runs whose outputs already exist, so an interrupted sweep resumes where it stopped.
# Resuming refuses to continue if the loaded database differs from the one the sweep started with.
RESUME = True

# "parquet" (falls back to pickle for a frame Arrow can't store) or "pickle"
SAVE_FORMAT = "parquet"

# The database is loaded ONCE and shared by all runs. If a sweep might be resumed on a later day,
# use the snapshot so the resumed runs see the same data.
use_reproducibility_snapshot = True

# Same modeling parameters as the main workbook
NUM_OPPS_FOR_TRAINING = 4
MIN_ARC_LENGTH_FOR_TRAINING = 20
MIN_NUM_OBS_FOR_TRAINING = 16
TRAINING_TIME_LIMIT = 2000

# Also vary AutoGluon's model seeds from run to run (in addition to the fold assignment).
VARY_MODEL_SEED = True

# False reproduces the main workbook: predictions come from the models refit on all data ("_FULL").
# True keeps the 8 fold models and averages them at inference instead.
USE_BAGGED_INFERENCE = False

# Delete each run's AutoGluon model directory after its outputs are saved.
DELETE_MODELS_AFTER_RUN = True

# Delta-H sweep settings (same as the main workbook)
H_NEUTRALITY_PROBABILITY_MIN = 0.8
H_NEUTRALITY_EXTENSION_DIFFICULTY_MAX = 0.1
H_NEUTRALITY_TOLERANCE = 0.5
H_GRID = np.arange(5.0, 30.01, 0.05)
H_NEUTRALITY_MAX_OBJECTS = 500
H_NEUTRALITY_MAX_OBJECTS_COMET = 200

extension_difficulty_threshold = 0.9

# Table 3 columns that get a ± (standard deviation across runs), and the decimals for mean and ±.
PM_COLUMNS = ["prob", "DeltaQ", "S_EV", "DeltaH"]
PM_DECIMALS = {"prob": 6, "DeltaQ": 1, "S_EV": 5, "DeltaH": 2, "exp_Num_opps": 1}
NOTES_CSV = "table_notes.csv"

In [13]:
with open('known_active_objects.json', 'r') as f:
    known_strongly_suspected_active_objects = json.load(f)

with open('dual_designation_list.json', 'r') as f:
    dual_designation_list = json.load(f)

discoveries_in_comet_file = ["P/2025 UX109 (Ye)","P/2023 JN16 (Lemmon)","P/2022 BV9 (Lemmon)","P/2017 FL36 (PANSTARRS)","489P/Denning"]

## Load the data once
Same loading, feature engineering, training filters and hard-coded single-opposition overrides as the main workbook. Nothing in this section is repeated per run.

In [14]:
if use_reproducibility_snapshot:
    orb = pd.read_parquet('orb_snapshot.parquet')
else:
    try:
        orb = utils.load_all_databases()
    except Exception as e:
        print(f"Error loading databases: {e}")
        cache_dir = os.path.join(os.getcwd(), '.cache')
        for filename in os.listdir(cache_dir):
            file_path = os.path.join(cache_dir, filename)
            try:
                if os.path.isfile(file_path):
                    os.unlink(file_path)
            except Exception as e:
                print(f"Error deleting file {file_path}: {e}")
        raise Exception(f"Failed to load databases. To fix this, try deleting all files in the cache directory: {cache_dir}. Original error: {e}")

# Pin the date the apparition features treat as "now" (days_since_last_opp, n_detectable, ...) to the
# MPCORB standard epoch, the most common Epoch in the database. Left unpinned, every feature_engineering
# call picks the latest Epoch in its own frame: the comet file (Epoch = perihelion time) lands around 2092,
# and each single-object Delta-H sweep frame uses that object's own epoch. The main workbook only avoids
# this because its pessimistic pass happens to call set_reference_epoch(2461200.5) and never resets it.
REFERENCE_EPOCH_JD = float(pd.to_numeric(orb["Epoch"], errors="coerce").mode().iloc[0])
utils.set_reference_epoch(REFERENCE_EPOCH_JD)
print(f"reference epoch for apparition features: JD {REFERENCE_EPOCH_JD}")

# Fingerprint of the loaded database, used to stop a resumed sweep from mixing in different data
database_fingerprint = {
    "reference_epoch_jd": REFERENCE_EPOCH_JD,
    "n_rows": int(len(orb)),
    "num_opps_sum": int(pd.to_numeric(orb["Num_opps"], errors="coerce").fillna(0).sum()),
    "num_obs_sum": int(pd.to_numeric(orb["Num_obs"], errors="coerce").fillna(0).sum()),
}
print(database_fingerprint)

orb = utils.feature_engineering(orb)

extension_difficulty = pd.read_csv("extension_difficulty.csv")
orb = orb.merge(extension_difficulty, on="Principal_desig", how="left")

{'n_rows': 1563218, 'num_opps_sum': 14293681, 'num_obs_sum': 527618579}


In [15]:
orb_training = orb[((orb["Arc_length"]>=MIN_ARC_LENGTH_FOR_TRAINING)|(orb["Arc_length"].isna())|(orb["Perihelion_dist"]<1.3))
                       &(orb["Num_obs"]>=MIN_NUM_OBS_FOR_TRAINING)
                       &~orb["filtered_out"].astype(bool)
                       &(orb["a_diff_abs"]<0.0005)
                       &(orb["e_diff_abs"]<0.00015)
                       &(orb["i_diff_abs"]<0.003)
                       &(orb["multi_opp_disagree"]==0)
                       &((orb["extension_difficulty"]<0.1)| (orb["extension_difficulty"].isna())|(orb["Perihelion_dist"]<1.3))
                       &((orb["U"]<9)|(orb["Perihelion_dist"]<1.3))
                       &~orb["Principal_desig"].isin(known_strongly_suspected_active_objects)
                       &~orb["Number"].isin(dual_designation_list)]

print(f"Total objects in orbit database: {len(orb)}")
print(f"Total objects in decent orbit database: {len(orb_training)}")

mlcols = ['H', 'Node', 'a', 'i', 'vis_q',
          'Perihelion_direction_x_e', 'Perihelion_direction_y_e',
          'days_since_last_opp',
          'n_detectable', 'n_detectable_var', 'years_since_first_detectable', 'days_since_last_detectable',
       'Is_Past_Threshold']
mlcols_reg = mlcols.copy()
mlcols_reg.remove("Is_Past_Threshold")
mlcols_reg.append("Num_opps_minus_one")

# Fold assignments are added per run
data_df = orb_training.dropna(subset=["H"])[mlcols].astype(np.float32)
data_df_reg = orb_training.dropna(subset=["H"])[mlcols_reg].astype(np.float32)

# Hard code both 2009 DP2 and 2024 XE22 to Num_opps = 1 with Arc_length of 22 (main workbook cell "Hard code both...")
orb.loc[orb["Principal_desig"]=="2009 DP2", "Num_opps"] = 1
orb.loc[orb["Principal_desig"]=="2024 XE22", "Num_opps"] = 1
orb.loc[orb["Principal_desig"]=="2009 DP2", "Arc_length"] = 22
orb.loc[orb["Principal_desig"]=="2024 XE22", "Arc_length"] = 22

# Prediction frame before any model output; copied at the start of every run
orb_pred_base = orb.dropna(subset=["H_MPC","H_astdys","H_jpl"], how='all').copy()
orb_pred_base = utils.apply_nights_overrides(orb_pred_base)
orb_pred_base = utils.apply_magnitude_corrections(orb_pred_base)

Total objects in orbit database: 1563218
Total objects in decent orbit database: 1377035


In [16]:
# Comet orbit file (main workbook "Import the Comet Orbit File from MPC"), feature-engineered once
comet = pd.read_json("allcometels.json.gz",compression='gzip')
comet = comet[comet["Orbit_type"].isin(["P","D","A"])]

comet["a"] = comet["Perihelion_dist"] / (1 - comet["e"])
comet["Orbital_period"] = np.sqrt(comet["a"]**3)


def calc_jd(year, month, day):
    y = year.copy()
    m = month.copy()
    mask = m <= 2
    y[mask] -= 1
    m[mask] += 12
    A = np.floor(y / 100)
    B = 2 - A + np.floor(A / 4)
    B[y < 1582] = 0
    B[(y == 1582) & (month < 10)] = 0
    B[(y == 1582) & (month == 10) & (day <= 4)] = 0
    return np.floor(365.25 * (y + 4716)) + np.floor(30.6001 * (m + 1)) + day + B - 1524.5

comet["Tp"] = calc_jd(comet["Year_of_perihelion"], comet["Month_of_perihelion"], comet["Day_of_perihelion"])
comet["Epoch"] = comet["Tp"]
comet["M"] = 0.0

comet_mags_fo = pd.read_fwf("./comet_orbits/comet_els_fo.txt", widths=[8, 5, 1000], names=["Packed", "Mag", "Discard"])

comet["fullpacked"] = comet["Orbit_type"] + comet["Provisional_packed_desig"]
comet.drop(columns=["H"], inplace=True, errors="ignore") # Wrong H! This is the comet H and not the inert asteroidal H
cometmerge_base = comet.merge(comet_mags_fo, left_on="fullpacked", right_on="Packed", how="left")

cometmerge_base.loc[cometmerge_base["Designation_and_name"] == "489P/Denning", "Mag"] = 15.35
cometmerge_base.loc[cometmerge_base["Designation_and_name"] == "P/2025 UX109 (Ye)", "Mag"] = 15.00

cometmerge_base["H"] = cometmerge_base["Mag"]
assert utils._reference_epoch(cometmerge_base) == REFERENCE_EPOCH_JD, "reference epoch not pinned; run the data-loading cell first"
cometmerge_base = utils.feature_engineering(cometmerge_base)

/Users/petervw/mambaforge/envs/actscope-16/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/petervw/GitHub/ActivitySCOPE/activityscope_utils.py:2029: RuntimeWarning: invalid value encountered in sqrt
  sqrt_1pe, sqrt_1me = col(np.sqrt(1.0 + e)), col(np.sqrt(1.0 - e))


In [17]:
# Table 3 objects. Every one of them gets a Delta-H in every run, whatever that run's candidate ranking.
table_notes = paper_table.load_notes(NOTES_CSV)


def notes_designation_to_principal(designation):
    """'362P/(457175) 2008 GO98' -> '2008 GO98' (comet prefix and number stripped)."""
    return re.sub(r"^\(\d+\)\s+", "", paper_table._COMET_PREFIX_RE.sub("", designation))


_orb_desigs = set(orb["Principal_desig"])
table_orbit_desigs = [d for d in (notes_designation_to_principal(r["designation"]) for r in table_notes) if d in _orb_desigs]
_comet_names = set(cometmerge_base["Designation_and_name"])
table_comet_names = [r["designation"] for r in table_notes if r["designation"] in _comet_names]
_unresolved = [r["designation"] for r in table_notes
               if notes_designation_to_principal(r["designation"]) not in _orb_desigs and r["designation"] not in _comet_names]
print(f"Table 3 objects: {len(table_orbit_desigs)} in the orbit database, {len(table_comet_names)} in the comet file")
if _unresolved:
    print("WARNING: not found in either source (no Delta-H will be forced for these):", _unresolved)

Table 3 objects: 19 in the orbit database, 5 in the comet file


## One run
Each step mirrors the main workbook; the section it comes from is named in the comments.

In [18]:
save_path = os.path.join(tempfile.gettempdir(), 'activityscope_ag_multi')

# There's a known issue where torch probability calibration dies sometimes with multiple threads. This is a workaround.
torch.set_num_threads(1)

DH_COLS = ["H_neutral", "H_change_to_neutral", "exp_Num_opps_at_neutral_H", "neutrality_residual", "neutral_within_0p5"]
TABLE_SOURCE_COLS = ["Object", "q", "a", "e", "i", "H", "TJ", "Num_opps", "exp_Num_opps", "prob",
                     "quantile_Opps", "DeltaQ", "poisson_cdf", "H_change_to_neutral"]
TABLE_VALUE_COLS = paper_table.DATA_COLUMNS + ["DeltaH"]


def fit_kwargs(seed):
    ensemble_args = {"fold_fitting_strategy": "sequential_local"}
    if VARY_MODEL_SEED:
        ensemble_args.update({"model_random_seed": int(seed), "vary_seed_across_folds": True})
    kwargs = dict(presets="good_quality", num_bag_folds=8, dynamic_stacking=False,
                  ag_args_ensemble=ensemble_args, time_limit=TRAINING_TIME_LIMIT/3)
    if USE_BAGGED_INFERENCE:
        kwargs.update(refit_full=False, set_best_to_refit_full=False, save_bag_folds=True)
    return kwargs


def with_oof(full_pred, oof_pred):
    """Predictions for every row, replaced by the out-of-fold prediction wherever the row was in training."""
    out = pd.Series(np.asarray(full_pred, dtype=float), index=full_pred.index)
    common = out.index.intersection(oof_pred.index)
    out.loc[common] = np.asarray(oof_pred.loc[common], dtype=float)
    return out


def getFinal(orb_pred):
    final = orb_pred.copy()
    final.rename(columns={"Perihelion_dist":"q","Aphelion_dist":"Q"},inplace=True)
    final.set_index("Principal_desig", inplace=True)
    final['DeltaQ'] = final["quantile_Opps"]-final["Num_opps"]
    final.sort_values("DeltaQ",ascending=False, inplace=True)
    final["Known / Strong Suspect"] = final.index.isin(known_strongly_suspected_active_objects)
    return final


def delta_h_sweep(candidates, predictor_reg, observed_num_opps=None):
    """Counterfactual H sweep from the main workbook: hold orbit and astrometry fixed, step H over H_GRID,
    and take the H closest to the catalog value with |E[N_opp] - N_opp| <= H_NEUTRALITY_TOLERANCE.
    observed_num_opps=None uses each candidate's Num_opps (orbit list); the comet list passes 1."""
    results = []
    for designation, candidate in candidates.iterrows():
        observed = float(candidate["Num_opps"]) if observed_num_opps is None else float(observed_num_opps)
        h_scenarios = pd.DataFrame([candidate] * len(H_GRID))
        h_scenarios["H"] = H_GRID
        h_scenarios = utils.feature_engineering(h_scenarios)
        swept_exp_num_opps = np.asarray(predictor_reg.predict(h_scenarios), dtype=float) + 1
        residuals = np.abs(swept_exp_num_opps - observed)
        neutral_indices = np.flatnonzero(residuals <= H_NEUTRALITY_TOLERANCE)
        if len(neutral_indices):
            best_index = neutral_indices[np.abs(H_GRID[neutral_indices] - candidate["H"]).argmin()]
        else:
            best_index = residuals.argmin()
        results.append({
            "Designation": designation,
            "H_neutral": H_GRID[best_index],
            "H_change_to_neutral": H_GRID[best_index] - candidate["H"],
            "exp_Num_opps_at_neutral_H": swept_exp_num_opps[best_index],
            "neutrality_residual": residuals[best_index],
            "neutral_within_0p5": residuals[best_index] <= H_NEUTRALITY_TOLERANCE,
        })
    return pd.DataFrame(results, columns=["Designation"] + DH_COLS).set_index("Designation")


def attach_delta_h(frame, dh, keys):
    """Copy the Delta-H columns onto frame (NaN for rows that weren't swept). keys aligns frame rows to dh's index."""
    for col in DH_COLS:
        frame[col] = pd.Series(keys, index=frame.index).map(dh[col])
    frame["neutral_within_0p5"] = frame["neutral_within_0p5"].astype("boolean")


def table_rows_for_run(final, cometmerge):
    """Table 3 values for one run, computed exactly as paper_table does (including the dagger single-opposition rows)."""
    for_paper = final[[c for c in ["Number"] + TABLE_SOURCE_COLS if c in final.columns]].copy()
    for_paper["Principal_desig"] = for_paper.index
    for_paper["Object"] = for_paper["Number"].astype(str) + " " + for_paper["Principal_desig"]
    for_paper["Object"] = for_paper["Object"].str.replace("<NA> ", "")

    comet = cometmerge.rename(columns={"Designation_and_name": "Object"})
    frames = [df[[c for c in TABLE_SOURCE_COLS if c in df.columns]] for df in (for_paper, comet)]
    combined = pd.concat(frames, ignore_index=True)
    combined["DeltaH"] = pd.to_numeric(combined["H_change_to_neutral"], errors="coerce")
    combined = paper_table._ensure_derived(combined)
    by_object = combined.drop_duplicates(subset="Object").set_index("Object")

    rows = []
    for r in table_notes:
        data = paper_table._lookup(by_object, r["designation"])
        found = data is not None
        if r["nopp_dagger"]:
            data = paper_table._as_single_opp(data)
        rows.append({"designation": r["designation"], "found": found,
                     **{c: paper_table._num(data, c) for c in TABLE_VALUE_COLS}})
    return pd.DataFrame(rows)


def save_frame(df, stem):
    if SAVE_FORMAT == "parquet":
        path = stem + ".parquet"
        try:
            df.to_parquet(path)
            return path
        except Exception as e:
            print(f"  parquet failed for {os.path.basename(stem)} ({type(e).__name__}: {e}); saving as pickle")
            if os.path.exists(path):
                os.remove(path)
    path = stem + ".pkl"
    df.to_pickle(path)
    return path

In [19]:
def run_once(run_idx):
    seed = BASE_SEED + run_idx
    stem = os.path.join(OUTPUT_DIR, f"run_{run_idx:03d}")
    run_model_path = os.path.join(save_path, f"{RUN_NAME}_run{run_idx:03d}_{datetime.datetime.now():%Y%m%d_%H%M%S}")
    timings = {}
    t0 = time.time()
    utils.set_reference_epoch(REFERENCE_EPOCH_JD)  # the Delta-H sweep frames would otherwise each pick their own epoch
    try:
        # ---- Modeling (main workbook "Modeling"), with this run's folds and seeds ----
        d_bin = data_df.copy()
        d_reg = data_df_reg.copy()
        shared_folds = np.random.default_rng(seed).integers(0, 8, size=len(d_bin))
        d_bin["Shared_Fold"] = shared_folds
        d_reg["Shared_Fold"] = shared_folds

        predictor_reg = TabularPredictor(label="Num_opps_minus_one", groups="Shared_Fold", eval_metric=utils.POISSON_SCORER, problem_type='regression', path=os.path.join(run_model_path, "reg"))
        predictor_reg.fit(d_reg, hyperparameters=utils.HYPERPARAMETERS_POISSON, num_stack_levels=0, **fit_kwargs(seed))
        d_bin["exp_Num_opps"] = with_oof(predictor_reg.predict(d_bin) + 1, TabularPredictor.predict_oof(predictor_reg) + 1)
        d_reg["exp_Num_opps"] = d_bin["exp_Num_opps"]

        predictor_quant = TabularPredictor(label="Num_opps_minus_one", groups="Shared_Fold", problem_type='quantile', quantile_levels=[0.006], path=os.path.join(run_model_path, "quant"))
        predictor_quant.fit(d_reg, hyperparameters=utils.HYPERPARAMETERS_QUANTILE, num_stack_levels=1, **fit_kwargs(seed))
        d_bin["quantile_Opps"] = with_oof(predictor_quant.predict(d_bin)[0.006] + 1, TabularPredictor.predict_oof(predictor_quant)[0.006] + 1)

        predictor = TabularPredictor(label="Is_Past_Threshold", groups="Shared_Fold", eval_metric='log_loss', path=os.path.join(run_model_path, "bin"))
        predictor.fit(d_bin, hyperparameters=utils.HYPERPARAMETERS_BINARY, num_stack_levels=0, **fit_kwargs(seed))
        timings["training_s"] = time.time() - t0

        # ---- Predictions (main workbook "Make Predictions") ----
        t = time.time()
        orb_pred = orb_pred_base.copy()
        orb_pred["exp_Num_opps"] = with_oof(predictor_reg.predict(orb_pred) + 1, TabularPredictor.predict_oof(predictor_reg) + 1)
        orb_pred["quantile_Opps"] = with_oof(predictor_quant.predict(orb_pred)[0.006] + 1, TabularPredictor.predict_oof(predictor_quant)[0.006] + 1)
        orb_pred["prob"] = with_oof(predictor.predict_proba(orb_pred)[1], TabularPredictor.predict_proba_oof(predictor)[1])
        orb_pred["poisson_cdf"] = poisson.cdf(orb_pred["Num_opps"]-1, orb_pred["exp_Num_opps"]-1)

        final = getFinal(orb_pred)

        # ---- Extension difficulty classifier, then drop filtered objects ----
        orb_pred, final = utils.train_extension_difficulty_classifier(orb_pred, final, orb)
        orb_pred = orb_pred[~orb_pred["filtered_out"].astype(bool)]
        final = final[~final["filtered_out"].astype(bool)].copy()
        timings["predictions_s"] = time.time() - t

        # ---- Centralized ranking (main workbook "Centralized Ranking") ----
        final["priority_score"] = final["DeltaQ"] - (final["extension_difficulty"] * 15.5)
        combined_rankings = final[(((final["Arc_length"]>=8)|(final["Arc_length"].isna()))
              &(final["Num_obs"]>=6)
              &((final["Num_obs"]>=7)|(final["nights_total"]>=5))
              &(final["DeltaQ"]>0.3*final["quantile_Opps"])
              &(final["prob"]>0.9)
              &(final["q"]>1.1)
              &((final["extension_difficulty"].between(0,extension_difficulty_threshold))|(final["nights_total"].isna()))
              &(final["Orbital_period"]<20))| final.index.isin(known_strongly_suspected_active_objects)].copy()
        combined_rankings.sort_values("priority_score", ascending=False, inplace=True)

        # ---- Delta-H, orbit database (main workbook "Counterfactual H sweep" on combined_rankings) ----
        t = time.time()
        h_candidates = combined_rankings[
            (combined_rankings["prob"] >= H_NEUTRALITY_PROBABILITY_MIN)
            & (combined_rankings["extension_difficulty"] <= H_NEUTRALITY_EXTENSION_DIFFICULTY_MAX)
            & combined_rankings["Num_opps"].notna()
        ].head(H_NEUTRALITY_MAX_OBJECTS)
        # also the known/strongly suspected active objects and every Table 3 object
        forced = final.loc[final.index.intersection(pd.Index(list(known_strongly_suspected_active_objects) + table_orbit_desigs))]
        h_candidates = pd.concat([h_candidates, forced])
        h_candidates = h_candidates[~h_candidates.index.duplicated(keep="first")]
        h_candidates = h_candidates[h_candidates["Num_opps"].notna() & h_candidates["H"].notna()]
        dh_orbit = delta_h_sweep(h_candidates, predictor_reg)
        attach_delta_h(final, dh_orbit, final.index)

        # ---- Comet file predictions and Delta-H (main workbook comet cells) ----
        cometmerge = cometmerge_base.copy()
        cometmerge["exp_Num_opps"] = predictor_reg.predict(cometmerge) + 1
        cometmerge["quantile_Opps"] = predictor_quant.predict(cometmerge)[0.006] + 1
        cometmerge["prob"] = predictor.predict_proba(cometmerge)[1]
        cometmerge["Num_opps"] = pd.NA # no data for this in the comet file
        cometmerge.loc[cometmerge["Designation_and_name"].isin(discoveries_in_comet_file), "Num_opps"] = 1
        cometmerge["Num_opps"] = cometmerge["Num_opps"].astype("Int64")
        num_opps = pd.to_numeric(cometmerge["Num_opps"], errors="coerce")
        exp_num_opps = pd.to_numeric(cometmerge["exp_Num_opps"], errors="coerce")
        cometmerge["poisson_cdf"] = pd.Series(poisson.cdf(num_opps - 1, exp_num_opps - 1), index=cometmerge.index, dtype="Float64")
        cometmerge['DeltaQ'] = cometmerge["quantile_Opps"]-cometmerge["Num_opps"]
        cometmerge.rename(columns={"Perihelion_dist":"q","Aphelion_dist":"Q"},inplace=True)

        comet_candidates = cometmerge[cometmerge["H"].notna()].sort_values("prob", ascending=False).head(H_NEUTRALITY_MAX_OBJECTS_COMET)
        comet_forced = cometmerge[cometmerge["Designation_and_name"].isin(set(discoveries_in_comet_file) | set(table_comet_names))
                                  & cometmerge["H"].notna()]
        comet_candidates = pd.concat([comet_candidates, comet_forced]).set_index("Designation_and_name", drop=False)
        comet_candidates = comet_candidates[~comet_candidates.index.duplicated(keep="first")]
        dh_comet = delta_h_sweep(comet_candidates, predictor_reg, observed_num_opps=1.0)
        attach_delta_h(cometmerge, dh_comet, cometmerge["Designation_and_name"])
        timings["delta_h_s"] = time.time() - t

        # ---- Save ----
        t = time.time()
        rows = table_rows_for_run(final, cometmerge)
        rows.insert(0, "run", run_idx)
        rows.insert(1, "seed", seed)
        paths = {"final": save_frame(final, stem + "_final"),
                 "cometmerge": save_frame(cometmerge, stem + "_cometmerge")}
        timings["save_s"] = time.time() - t
        timings["total_s"] = time.time() - t0

        meta = {
            "run": run_idx, "seed": seed, "finished": datetime.datetime.now().isoformat(timespec="seconds"),
            "timings_s": timings, "files": paths,
            "n_delta_h_orbit": int(len(dh_orbit)), "n_delta_h_comet": int(len(dh_comet)),
            "model_best": {name: p.model_best for name, p in [("reg", predictor_reg), ("quant", predictor_quant), ("bin", predictor)]},
            "score_val": {name: p.leaderboard(silent=True).set_index("model")["score_val"].dropna().to_dict()
                          for name, p in [("reg", predictor_reg), ("quant", predictor_quant), ("bin", predictor)]},
        }
        with open(stem + "_meta.json", "w") as f:
            json.dump(meta, f, indent=2, default=str)
        rows.to_csv(stem + "_table_rows.csv", index=False)  # written last: marks the run complete for RESUME

        missing = rows.loc[~rows["found"], "designation"].tolist()
        if missing:
            print(f"  WARNING: Table 3 objects with no data this run: {missing}")
        return rows
    finally:
        gc.collect()
        if DELETE_MODELS_AFTER_RUN:
            shutil.rmtree(run_model_path, ignore_errors=True)

## Run the sweep

In [20]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

params_path = os.path.join(OUTPUT_DIR, "sweep_parameters.json")
sweep_parameters = {
    "N_RUNS": N_RUNS, "BASE_SEED": BASE_SEED, "use_reproducibility_snapshot": use_reproducibility_snapshot,
    "TRAINING_TIME_LIMIT": TRAINING_TIME_LIMIT, "VARY_MODEL_SEED": VARY_MODEL_SEED,
    "USE_BAGGED_INFERENCE": USE_BAGGED_INFERENCE, "H_NEUTRALITY_MAX_OBJECTS": H_NEUTRALITY_MAX_OBJECTS,
    "H_NEUTRALITY_MAX_OBJECTS_COMET": H_NEUTRALITY_MAX_OBJECTS_COMET, "database_fingerprint": database_fingerprint,
}
if os.path.exists(params_path) and glob.glob(os.path.join(OUTPUT_DIR, "run_*_table_rows.csv")):
    with open(params_path) as f:
        previous = json.load(f)
    if previous.get("database_fingerprint") != database_fingerprint:
        raise RuntimeError(
            f"{OUTPUT_DIR} already has runs made on a different database ({previous.get('database_fingerprint')} vs "
            f"{database_fingerprint}). Use a new RUN_NAME, or load the same data, so runs aren't mixed.")
with open(params_path, "w") as f:
    json.dump(sweep_parameters, f, indent=2, default=str)

sweep_start = time.time()
for run_idx in range(N_RUNS):
    if RESUME and os.path.exists(os.path.join(OUTPUT_DIR, f"run_{run_idx:03d}_table_rows.csv")):
        print(f"run {run_idx}: already complete, skipping")
        continue
    print(f"\n===== run {run_idx + 1} of {N_RUNS} (seed {BASE_SEED + run_idx}), started {datetime.datetime.now():%Y-%m-%d %H:%M} =====")
    t = time.time()
    try:
        run_once(run_idx)
        print(f"===== run {run_idx + 1} of {N_RUNS} done in {(time.time() - t) / 60:.1f} min =====")
    except Exception:
        # Keep going so one failure doesn't cost the rest of the night; the traceback is saved next to the outputs.
        err = traceback.format_exc()
        print(err)
        with open(os.path.join(OUTPUT_DIR, f"run_{run_idx:03d}_error.txt"), "w") as f:
            f.write(err)
print(f"\nsweep finished in {(time.time() - sweep_start) / 3600:.2f} h")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.11.16
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.6.0: Tue Jul 21 20:51:05 PDT 2026; root:xnu-11417.140.69.711.44~1/RELEASE_ARM64_T6000
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       11.65 GB / 32.00 GB (36.4%)
Disk Space Avail:   36.06 GB / 926.35 GB (3.9%)
Presets specified: ['good_quality']
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting `save_bag_folds=True`.



===== run 1 of 2 (seed 1000), started 2026-09-15 07:33 =====


Values in column 'Shared_Fold' used as split folds instead of being automatically set. Bagged models will have 8 splits.
Beginning AutoGluon training ... Time limit = 667s
AutoGluon will save models to "/var/folders/q_/q72gghqx0dbbrygb127lyk2m0000gp/T/activityscope_ag_multi/table3_seed_sweep_run000_20260915_073339/reg"
Train Data Rows:    1377032
Train Data Columns: 13
Label Column:       Num_opps_minus_one
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    11991.46 MB
	Train Data (Original)  Memory Usage: 63.04 MB (0.5% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4

[1000]	valid_set's poisson: -13.2477	valid_set's mean_poisson_deviance: -0.383516
[1000]	valid_set's poisson: -13.2117	valid_set's mean_poisson_deviance: -0.388538
[1000]	valid_set's poisson: -13.2132	valid_set's mean_poisson_deviance: -0.384733
[1000]	valid_set's poisson: -13.3092	valid_set's mean_poisson_deviance: -0.384205
[1000]	valid_set's poisson: -13.2695	valid_set's mean_poisson_deviance: -0.382975
[1000]	valid_set's poisson: -13.1957	valid_set's mean_poisson_deviance: -0.384636
[1000]	valid_set's poisson: -13.2664	valid_set's mean_poisson_deviance: -0.385501
[1000]	valid_set's poisson: -13.2095	valid_set's mean_poisson_deviance: -0.381962


	-0.3845	 = Validation score   (-mean_poisson_deviance)
	152.21s	 = Training   runtime
	53.91s	 = Validation runtime
Fitting model: XGBoost_BAG_L1 ... Training model for up to 458.21s of the 458.21s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=10, gpus=0)
	-0.3849	 = Validation score   (-mean_poisson_deviance)
	431.92s	 = Training   runtime
	9.24s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.00s of the 16.05s of remaining time.
	Fitting 1 model on all data | Fitting with cpus=10, gpus=0, mem=0.1/9.2 GB
	Ensemble Weights: {'LightGBM_BAG_L1': 0.556, 'XGBoost_BAG_L1': 0.444}
	-0.3833	 = Validation score   (-mean_poisson_deviance)
	0.56s	 = Training   runtime
	0.01s	 = Validation runtime
AutoGluon training complete, total runtime = 651.77s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 2725.7 rows/s (172129 batch size)
Automatically performing 

...

===== run 1 of 2 done in 39.4 min =====

===== run 2 of 2 (seed 1001), started 2026-09-15 08:13 =====


Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    9423.63 MB
	Train Data (Original)  Memory Usage: 63.04 MB (0.7% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', []) : 12 | ['H', 'Node', 'a', 'i', 'vis_q', ...]
	Types of features in processed data (raw dtype, special dtypes):
		('float', []) : 12 | ['H', 'Node', 'a', 'i', 'vis_q', ...]
	1.2s = Fit runtime
	12 features in original data used to generate 12 features in processed da

[1000]	valid_set's poisson: -13.2185	valid_set's mean_poisson_deviance: -0.387417
[1000]	valid_set's poisson: -13.2136	valid_set's mean_poisson_deviance: -0.383453
[1000]	valid_set's poisson: -13.28	valid_set's mean_poisson_deviance: -0.383238
[1000]	valid_set's poisson: -13.2507	valid_set's mean_poisson_deviance: -0.386229
[1000]	valid_set's poisson: -13.2327	valid_set's mean_poisson_deviance: -0.384697
[1000]	valid_set's poisson: -13.1976	valid_set's mean_poisson_deviance: -0.381936
[1000]	valid_set's poisson: -13.2354	valid_set's mean_poisson_deviance: -0.383878
[1000]	valid_set's poisson: -13.2945	valid_set's mean_poisson_deviance: -0.384331


	-0.3844	 = Validation score   (-mean_poisson_deviance)
	151.63s	 = Training   runtime
	53.7s	 = Validation runtime
Fitting model: XGBoost_BAG_L1 ... Training model for up to 459.09s of the 459.09s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=10, gpus=0)
	-0.3847	 = Validation score   (-mean_poisson_deviance)
	432.77s	 = Training   runtime
	9.29s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.00s of the 16.01s of remaining time.
	Fitting 1 model on all data | Fitting with cpus=10, gpus=0, mem=0.1/8.6 GB
	Ensemble Weights: {'LightGBM_BAG_L1': 0.545, 'XGBoost_BAG_L1': 0.455}
	-0.3831	 = Validation score   (-mean_poisson_deviance)
	0.62s	 = Training   runtime
	0.01s	 = Validation runtime
AutoGluon training complete, total runtime = 651.55s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 2732.6 rows/s (172129 batch size)
Automatically performing r

Traceback (most recent call last):
  File "/var/folders/q_/q72gghqx0dbbrygb127lyk2m0000gp/T/ipykernel_53756/4115635105.py", line 28, in <module>
    run_once(run_idx)
  File "/var/folders/q_/q72gghqx0dbbrygb127lyk2m0000gp/T/ipykernel_53756/2470788017.py", line 21, in run_once
    predictor_quant.fit(d_reg, hyperparameters=utils.HYPERPARAMETERS_QUANTILE, num_stack_levels=1, **fit_kwargs(seed))
  File "/Users/petervw/mambaforge/envs/actscope-16/lib/python3.11/site-packages/autogluon/common/utils/decorators.py", line 34, in _call
    return f(*gargs, **gkwargs)
           ^^^^^^^^^^^^^^^^^^^^
  File "/Users/petervw/mambaforge/envs/actscope-16/lib/python3.11/site-packages/autogluon/tabular/predictor/predictor.py", line 1596, in fit
    self._fit(ag_fit_kwargs=ag_fit_kwargs, ag_post_fit_kwargs=ag_post_fit_kwargs)
  File "/Users/petervw/mambaforge/envs/actscope-16/lib/python3.11/site-packages/autogluon/tabular/predictor/predictor.py", line 1604, in _fit
    self._post_fit(**ag_post_fit_kwarg

## Aggregate the runs
Everything below reads only the saved `run_XXX_table_rows.csv` files, so it can be re-run at any time (including while a sweep is still going, or after restarting the kernel once the parameter and data-loading cells above have run).

In [21]:
run_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "run_*_table_rows.csv")))
if not run_files:
    raise RuntimeError(f"No completed runs in {OUTPUT_DIR}")
all_runs = pd.concat([pd.read_csv(f) for f in run_files], ignore_index=True)
n_runs_done = all_runs["run"].nunique()
print(f"{n_runs_done} completed runs: {sorted(all_runs['run'].unique().tolist())}")

table_order = [r["designation"] for r in table_notes]
stats = (all_runs.groupby("designation")[TABLE_VALUE_COLS]
         .agg(["mean", "std", "median", "min", "max", "count"])
         .reindex(table_order))
stats.to_csv(os.path.join(OUTPUT_DIR, "table3_run_statistics.csv"))

summary = pd.DataFrame(index=table_order)
for c in TABLE_VALUE_COLS:
    summary[c] = stats[(c, "mean")]
    if c in PM_COLUMNS:
        summary[c + " ±"] = stats[(c, "std")]
display(summary)

not_found = all_runs.loc[~all_runs["found"].astype(bool)]
if len(not_found):
    print("Objects missing in some runs:")
    display(not_found.groupby("designation")["run"].apply(list))

1 completed runs: [0]


,q,a,e,i,TJ,H,Num_opps,exp_Num_opps,prob,prob ±,DeltaQ,DeltaQ ±,S_EV,S_EV ±,DeltaH,DeltaH ±
2010 RH69,4.276048,4.532396,0.056559,11.54822,2.973910,13.95,3.0,18.461872,0.999993,NaN,11.916621,NaN,4.458606e-06,NaN,2.45,NaN
362P/(457175) 2008 GO98,2.864301,3.971173,0.278727,15.55983,2.926730,12.91,7.0,24.189320,0.999993,NaN,10.013975,NaN,2.430997e-05,NaN,3.24,NaN
282P/(323137) 2003 BM80,3.438718,4.234372,0.187904,5.81319,2.991756,13.62,7.0,22.440346,0.999995,NaN,10.013975,NaN,8.951644e-05,NaN,2.13,NaN
2017 QN84,2.474881,3.769087,0.343374,12.07147,2.943831,14.97,4.0,12.877176,0.999995,NaN,5.017560,NaN,2.519445e-03,NaN,2.08,NaN
2008 VK110,4.436583,7.167172,0.380986,7.38588,2.878252,13.75,3.0,7.629636,0.992965,NaN,-0.149289,NaN,3.909854e-02,NaN,1.70,NaN
2001 BV70,2.063079,3.593632,0.425907,4.33032,2.947403,15.14,1.0,16.180550,0.999996,NaN,10.939054,NaN,2.553707e-07,NaN,4.16,NaN
P/2025 UX109 (Ye),2.569961,3.806430,0.324837,3.18090,2.982290,15.00,1.0,22.442198,0.999996,NaN,16.013975,NaN,4.872724e-10,NaN,4.20,NaN
2019 OE31,3.942522,4.381322,0.100152,5.22115,3.006031,14.64,1.0,16.873985,0.999995,NaN,11.843466,NaN,1.276485e-07,NaN,3.26,NaN
2018 BJ11,3.252873,4.200594,0.225616,3.43068,2.986205,15.55,1.0,10.415681,0.999983,NaN,4.913115,NaN,8.143700e-05,NaN,3.30,NaN
P/2022 BV9 (Lemmon),3.347685,4.366555,0.233335,11.92230,2.934749,13.79,1.0,23.923008,0.999991,NaN,16.013975,NaN,1.108317e-10,NaN,4.41,NaN


In [22]:
# Per-run values for each ± column, to see where the scatter comes from
for c in PM_COLUMNS:
    print(c)
    display(all_runs.pivot(index="designation", columns="run", values=c).reindex(table_order))

prob


run,0
designation,
2010 RH69,0.999993
362P/(457175) 2008 GO98,0.999993
282P/(323137) 2003 BM80,0.999995
2017 QN84,0.999995
2008 VK110,0.992965
2001 BV70,0.999996
P/2025 UX109 (Ye),0.999996
2019 OE31,0.999995
2018 BJ11,0.999983


DeltaQ


run,0
designation,
2010 RH69,11.916621
362P/(457175) 2008 GO98,10.013975
282P/(323137) 2003 BM80,10.013975
2017 QN84,5.017560
2008 VK110,-0.149289
2001 BV70,10.939054
P/2025 UX109 (Ye),16.013975
2019 OE31,11.843466
2018 BJ11,4.913115


S_EV


run,0
designation,
2010 RH69,4.458606e-06
362P/(457175) 2008 GO98,2.430997e-05
282P/(323137) 2003 BM80,8.951644e-05
2017 QN84,2.519445e-03
2008 VK110,3.909854e-02
2001 BV70,2.553707e-07
P/2025 UX109 (Ye),4.872724e-10
2019 OE31,1.276485e-07
2018 BJ11,8.143700e-05


DeltaH


run,0
designation,
2010 RH69,2.45
362P/(457175) 2008 GO98,3.24
282P/(323137) 2003 BM80,2.13
2017 QN84,2.08
2008 VK110,1.70
2001 BV70,4.16
P/2025 UX109 (Ye),4.20
2019 OE31,3.26
2018 BJ11,3.30


In [24]:
# LaTeX Table 3: match committed table3.tex's text, symbols, and layout; update values with mean ± standard deviation.
old_order = True  # same meaning as in paper_table.build_table

PM_LATEX = {"prob": r"$P(N_{\rm opp}\ge4)$", "DeltaQ": r"$\Delta Q$", "S_EV": r"$S_{\rm EV}$",
            "DeltaH": r"$\Delta H$", "exp_Num_opps": r"$E[N_{\rm opp}]$"}
H_MARKERS = {"2008 VK110": r"$^{\S}$", "2009 DP2": r"$^{\S}$"}
COMMITTED_ORBIT_VALUES = {"2010 RH69": ["3.90", "4.27", "0.087", "11.92"]}


def fmt_value(col, mean, std, count):
    if col in PM_DECIMALS:
        text = "" if pd.isna(mean) else f"{mean:.{PM_DECIMALS[col]}f}"
    else:
        text = paper_table.FORMATTERS[col](mean)
    if col in PM_COLUMNS and text and count > 1 and not pd.isna(std):
        text += rf" $\pm$ {std:.{PM_DECIMALS.get(col, 2)}f}"
    return text


def render_row(r, s):
    obj = paper_table.format_designation(r["designation"], r["confirmed"]) + r["marker"]
    cells = [fmt_value(c, s[(c, "mean")], s[(c, "std")], s[(c, "count")]) for c in TABLE_VALUE_COLS]
    if r["designation"] in COMMITTED_ORBIT_VALUES:
        cells[:4] = COMMITTED_ORBIT_VALUES[r["designation"]]
    h_index = TABLE_VALUE_COLS.index("H")
    cells[h_index] = (cells[h_index] or "") + H_MARKERS.get(r["designation"], "")
    if r["nopp_dagger"]:
        i = TABLE_VALUE_COLS.index("Num_opps")
        cells[i] = (cells[i] or "") + paper_table._DAGGER
    cells = [c if c else "  " for c in cells]
    return f"\\objectcell{{{obj}}} &\n" + " & ".join(cells) + f" &\n{r['note']} \\\\\\"


# paper_table's header/footer plus the Delta-H column (as in table3.tex) and a note on the ± values
header = paper_table._HEADER
header = header.replace("{lrrrrrrrrrrrl}", "{lrrrrrrrrrrrrl}")
linebreak = "\\" * 2
header = header.replace(r"Note " + linebreak, r"$\Delta H$ &" + "\n" + r"Note " + linebreak)
header = header.replace(r"$S_{\rm EV}$\textsuperscript{g} & Note " + linebreak,
                        r"$S_{\rm EV}$\textsuperscript{g} & $\Delta H$ & Note " + linebreak)
header = header.replace(r"\multicolumn{13}", r"\multicolumn{14}")

footer = paper_table._FOOTER.replace(r"\multicolumn{13}", r"\multicolumn{14}")
committed_h_note = (r"\multicolumn{14}{l}{\footnotesize $^{\S}$\,\revision{A multi-opposition object with significant differences in $H_V$ across oppositions. We use the brightest $H_V$.}}"
                    + linebreak + "\n")
footer = footer.replace(r"\printtabnotes", committed_h_note + r"\printtabnotes")
pm_names = [PM_LATEX.get(c, c) for c in PM_COLUMNS]
pm_list = ", ".join(pm_names[:-1]) + ", and " + pm_names[-1] if len(pm_names) > 1 else pm_names[0]
pm_note = (r"\multicolumn{14}{l}{\footnotesize " + pm_list + r": mean $\pm$ standard deviation over "
           + f"{n_runs_done}" + r" training runs with different cross-validation folds and model seeds.}" + linebreak + "\n")
assert footer.count(r"\printtabnotes") == 1
footer = footer.replace(r"\printtabnotes", pm_note + r"\printtabnotes")

multi, single, asteroid = [], [], []
for r in table_notes:
    s = stats.loc[r["designation"]]
    if r["group"] == "cometary":
        nopp = s[("Num_opps", "mean")]
        (multi if (not pd.isna(nopp) and nopp > 1) else single).append((r, s))
    else:
        asteroid.append((r, s))

if not old_order:
    multi.sort(key=lambda e: paper_table._key_asc(e[1][("S_EV", "mean")]))
    single.sort(key=lambda e: paper_table._key_desc(e[1][("prob", "mean")]))
    asteroid.sort(key=lambda e: paper_table._key_desc(e[1][("prob", "mean")]))

parts = [header]
for label, entries, sortcap, newpage in [
    (paper_table.SECTION_MULTI_COMET, multi, paper_table._SORT_SEV_ASC, False),
    (paper_table.SECTION_SINGLE_COMET, single, paper_table._SORT_PROB_DESC, False),
    (paper_table.SECTION_ASTEROID, asteroid, paper_table._SORT_PROB_DESC, True),
]:
    if not entries:
        continue
    if newpage:
        parts.append(r"\newpage")
    parts.append(f"\\multicolumn{{14}}{{l}}{{\\textbf{{{label}}} \\textnormal{{\\footnotesize ({sortcap})}}}}\\\\")
    parts.append(r"\midrule")
    parts.append("")
    for r, s in entries:
        parts.append(render_row(r, s))
        parts.append("")
    parts.append(r"\midrule")
if parts[-1] == r"\midrule":
    parts.pop()
parts.append(footer)
latex_multi = "\n".join(parts)

table_path = os.path.join(OUTPUT_DIR, "table3_multi_run.tex")
with open(table_path, "w") as f:
    f.write(latex_multi)
print(f"written to {table_path}\n")
print(latex_multi)

written to multi_run_outputs/table3_seed_sweep/table3_multi_run.tex

\begin{longrotatetable}

\begin{longtable}{lrrrrrrrrrrrrl}
\caption{\activityscope\ discoveries, independent recoveries, and candidates. Objects with observed activity are marked with an asterisk.}
\label{tab:activityscope_results}\\
\toprule
Object & $q$ {\footnotesize (au)} & $a$ {\footnotesize (au)} & $e$ & $i~(^\circ)$ &
$T_J$\tabnote{Tisserand parameter with respect to Jupiter.} &
$H_V$\tabnote{Absolute magnitude in the $V$ band, assuming an inert object.} &
$N_{\rm opp}$\tabnote{Number of oppositions on which the object has been observed.} &
{\footnotesize $E[N_{\rm opp}]$}\tabnote{Expected number of observed oppositions for an inert object of the same orbit and $H_V$, predicted by the regression model.} &
{\footnotesize $P(N_{\rm opp}\ge4)$}\tabnote{Binary-classifier probability that an inert object of the same orbit and $H_V$ would be observed on at least four oppositions.} &
$\Delta Q$\tabnote{Quantile defici